# Etapa 2 — Aprendizado linear

Nas regressões lineares, estimamos a quantidade de equipamentos para alunos por sala utilizada nas escolas do Amazonas. Na logística, voltamos à pergunta da árvore: a escola tem infraestrutura digital básica?

In [ ]:
import subprocess
import sys
from pathlib import Path

Path("gráficos").mkdir(exist_ok=True)

def executar(script, *argumentos):
    resultado = subprocess.run([sys.executable, "-X", "utf8", "scripts/" + script, *argumentos],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, encoding="utf-8")
    print(resultado.stdout)
    resultado.check_returncode()

## Regressão univariada — ajustamos uma reta usando uma pista por vez. O R² de teste indica quanto da variação do alvo a reta explica.

In [2]:
executar('regressao_univariada.py', 'dados_etapa2.csv', '--grafico', 'gráficos/grafico_univariada.png')

Base: 5526 linhas | treino: 4421 | teste: 1105 | alvo: y (bruto)

                   feature  theta0  theta1  r_pearson  mse_treino  mse_teste  rmse_teste  r2_treino  r2_teste
        IN_INTERNET_ALUNOS  0.2776  2.2107     0.4748      2.5941     2.0311      1.4252     0.2254    0.2372
IN_LABORATORIO_INFORMATICA  0.2976  2.3636     0.4853      2.5603     2.0976      1.4483     0.2355    0.2122
               QT_EQUIP_TV  0.3985  0.2449     0.3430      2.9548     2.2886      1.5128     0.1177    0.1404
       QT_EQUIP_MULTIMIDIA  0.4741  0.2004     0.3031      3.0413     2.3041      1.5179     0.0919    0.1346
  IN_INTERNET_APRENDIZAGEM  0.2566  1.0219     0.2767      3.0925     2.4640      1.5697     0.0766    0.0746
    IN_EQUIP_LOUSA_DIGITAL  0.6718  1.5587     0.1128      3.3063     2.6182      1.6181     0.0127    0.0166

Modelo: ŷ = 0.2776 + 2.2107 · IN_INTERNET_ALUNOS   (R² teste = 0.2372)
Conferência com scikit-learn: |Δθ0| = 6.88e-15, |Δθ1| = 3.60e-14
Gráfico salvo em gráficos/g

![Resultado do modelo](gráficos/grafico_univariada.png)

## Regressão multivariada — combinamos as pistas e ajustamos os pesos por gradiente descendente. A curva mostra como o custo varia durante o treino.

In [3]:
executar('regressao_multivariada_gd.py', 'dados_etapa2.csv', '--grafico', 'gráficos/convergencia_gd.png')

Base: 5526 linhas | treino: 4421 | teste: 1105 | alvo: y (bruto)
alpha = 0.1 | iterações realizadas: 163 | J inicial = 1.9192 -> J final = 1.1447

                 parametro  theta_GD  theta_eq_normal  |diferenca|
                intercepto  0.014041         0.014034     0.000007
        IN_INTERNET_ALUNOS  1.093991         1.093884     0.000107
IN_LABORATORIO_INFORMATICA  1.376792         1.376916     0.000125
  IN_INTERNET_APRENDIZAGEM  0.291698         0.291724     0.000026
               QT_EQUIP_TV  0.079129         0.079132     0.000003
       QT_EQUIP_MULTIMIDIA  0.014025         0.014016     0.000009
    IN_EQUIP_LOUSA_DIGITAL  0.168362         0.168417     0.000055

Treino: MSE = 2.2893 | R² = 0.3164
Teste : MSE = 1.8030 | RMSE = 1.3428 | R² = 0.3228
Comparação no mesmo teste: R² GD = 0.3228 | R² LinearRegression = 0.3228

Gráfico salvo em gráficos/convergencia_gd.png



![Resultado do modelo](gráficos/convergencia_gd.png)

## Ridge — aumentar a penalização λ reduz a magnitude dos pesos. Cada linha do gráfico representa uma pista.

In [4]:
executar('regressao_ridge.py', 'dados_etapa2.csv', '--saida-grafico', 'gráficos/ridge_coeficientes.png')

Validação (GD com L2 vs. solução fechada), diferença máxima entre θ:
  λ =       0: 5.02e-07
  λ =      10: 5.09e-07
  λ =    1000: 2.95e-07

λ ótimo por CV (5 dobras): 77.43  (MSE CV = 2.32503)

         λ    MSE teste   R² teste
         0      1.80304     0.3228
     77.43      1.80064     0.3237  <- ótimo CV
       100      1.80000     0.3239
      1000      1.78942     0.3279
     1e+04      1.97160     0.2595

Coeficientes (variáveis padronizadas):
                 preditora  theta_OLS (λ=0)  theta_Ridge (λ=77.4)
        IN_INTERNET_ALUNOS          0.42991               0.42534
IN_LABORATORIO_INFORMATICA          0.51733               0.50911
  IN_INTERNET_APRENDIZAGEM          0.14457               0.14488
               QT_EQUIP_TV          0.20282               0.20205
       QT_EQUIP_MULTIMIDIA          0.03880               0.04356
    IN_EQUIP_LOUSA_DIGITAL          0.02231               0.02243

Gráfico salvo em gráficos/ridge_coeficientes.png



![Resultado do modelo](gráficos/ridge_coeficientes.png)

## Regressão logística — usamos as pistas da Etapa 1 para estimar a probabilidade da classe 1. Com limiar de 0,5, o modelo acerta cerca de 88% no teste, mas identifica só cerca de metade das escolas da classe 1.

In [5]:
executar('regressao_logistica.py', 'dados_etapa1.csv', '--grafico', 'gráficos/custo_logistica.png')

Base: 5562 linhas | treino: 4450 | teste: 1112 | positivos: 18.1% (estratificado)
alpha = 1.0 | iterações: 37421 | J: 0.6931 -> 0.3243

                 parametro   theta  odds_ratio
                intercepto -1.7718      0.1700
IN_LABORATORIO_INFORMATICA  2.6363     13.9620
             IN_BIBLIOTECA  0.4156      1.5153
 IN_ENERGIA_GERADOR_FOSSIL -0.3452      0.7081
                     RURAL -0.9921      0.3708
              DEP_ESTADUAL -0.9279      0.3954
               DEP_FEDERAL  2.7637     15.8579
               DEP_PRIVADA  0.2171      1.2425

Conferência com scikit-learn: diferença máxima em θ = 1.78e-03

Treino (limiar 0.5): acurácia = 0.8676 | precisão = 0.6641 | recall = 0.5447 | F1 = 0.5985 | AUC = 0.8474
  matriz de confusão [[TN, FP], [FN, TP]] = [[3422, 222], [367, 439]]

Teste (limiar 0.5): acurácia = 0.8840 | precisão = 0.7687 | recall = 0.5124 | F1 = 0.6149 | AUC = 0.8358
  matriz de confusão [[TN, FP], [FN, TP]] = [[880, 31], [98, 103]]

Referência: classificar tu

![Resultado do modelo](gráficos/custo_logistica.png)

## Comparação com scikit-learn
As saídas acima mostram a comparação com LinearRegression nas regressões simples e multivariada e com LogisticRegression na classificação. Na multivariada, os dois modelos usam as mesmas escolas para treino e teste.